# Deep Learning 017 — Backpropagation, Part 3: The "Why"

Companion notebook to the lesson. Part 1 gave the steps and Part 2 the mechanics. This one
answers the question underneath both: **why does subtracting the gradient make the loss go
down?**

Five ideas, each one checkable:

1. The loss is a function of *all* the parameters at once.
2. The gradient is the vector of partial derivatives.
3. A derivative is a rate of change — "if I nudge this, how much does that move?"
4. A derivative *at a point* is a number, and its **sign** is the useful part.
5. Minimising means walking against the sign, and the step size is the learning rate.

`numpy` only; one optional `matplotlib` cell.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## Concept 1 — the loss is a function of every parameter

Not of the data. The data is fixed once you have it; the only things that can change are
the weights. So the loss is a surface over parameter space, and training is a search on
that surface.

Take the smallest interesting case: one weight and one bias, MSE over a small dataset.

In [ ]:
X = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
Y = np.array([2.1, 4.2, 5.9, 8.1, 9.8])          # roughly y = 2x

def L(w, b):
    return np.mean((Y - (w * X + b)) ** 2)

print(f"L(0, 0)   = {L(0, 0):.4f}      <- terrible")
print(f"L(2, 0)   = {L(2, 0):.4f}      <- close to the truth")
print(f"L(2, 0.5) = {L(2, 0.5):.4f}")
print(f"L(5, 0)   = {L(5, 0):.4f}      <- worse than useless")

Four points on a surface. Training is finding its lowest point, without ever being able to
see the whole surface — only the slope where you are standing.

## Concepts 2 and 3 — the gradient is the vector of partial derivatives

$$\nabla L = \left[\frac{\partial L}{\partial w},\ \frac{\partial L}{\partial b}\right]$$

Each partial answers one question: *if I increase this parameter slightly and hold the
others still, how much does the loss change?*

In [ ]:
def grad(w, b, h=1e-6):
    dw = (L(w + h, b) - L(w - h, b)) / (2 * h)
    db = (L(w, b + h) - L(w, b - h)) / (2 * h)
    return np.array([dw, db])

for w, b in [(0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)]:
    g = grad(w, b)
    print(f"at w={w:>4.1f}, b={b:>4.1f}   loss {L(w, b):>8.4f}   gradient {np.round(g, 4)}")

## Concept 4 — the **sign** is what matters

Read the `dw` column above.

- At `w = 0` the derivative is **negative**: increasing `w` *decreases* the loss.
- At `w = 3` the derivative is **positive**: increasing `w` *increases* the loss.
- Somewhere between them it passes through zero. That is the minimum.

So the rule "step **against** the derivative" is not a convention, it is the definition of
the derivative's sign read backwards. Check it directly: nudge each parameter *with* and
*against* its gradient and see which direction actually lowers the loss.

In [ ]:
w, b, step = 0.0, 0.0, 0.005
g = grad(w, b)
print(f"start          loss {L(w, b):.6f}   gradient {np.round(g, 4)}")
print(f"step WITH  g   loss {L(w + step * g[0], b + step * g[1]):.6f}   <- worse")
print(f"step AGAINST g loss {L(w - step * g[0], b - step * g[1]):.6f}   <- better")

That single cell is the whole "why". **The gradient points uphill, so subtracting it walks
downhill.** Everything else in training is bookkeeping around this fact.

## Concept 5 — the learning rate is how far, not which way

The direction is settled. The step size is a separate and genuinely difficult choice, and
the failure modes at each end are easy to produce.

In [ ]:
def run(lr, steps=60, start=(0.0, 0.0)):
    w, b = start
    hist = [L(w, b)]
    for _ in range(steps):
        g = grad(w, b)
        w, b = w - lr * g[0], b - lr * g[1]
        hist.append(L(w, b))
        if not np.isfinite(hist[-1]) or hist[-1] > 1e12:
            break
    return np.array(hist), (w, b)

print(f"{'lr':>8}{'loss after 60 steps':>22}{'final w':>11}{'verdict':>16}")
for lr in (0.0001, 0.001, 0.01, 0.05, 0.08, 0.09, 0.1, 0.12):
    hist, (w, b) = run(lr)
    end = hist[-1]
    verdict = ("diverged" if not np.isfinite(end) or end > 1e6
               else "too slow" if end > 0.5 else "good")
    shown = f"{end:.6f}" if np.isfinite(end) and end < 1e6 else "blew up"
    print(f"{lr:>8}{shown:>22}{w:>11.3f}{verdict:>16}")

Both ends fail, and they fail differently.

- **Too small** — the direction is right and you never arrive. At `lr = 0.0001` sixty steps
  take the loss from 43.70 only as far as 32.89.
- **Too large** — each step overshoots the minimum and lands further up the other side, so
  the loss grows every step until it overflows. On this data that happens between 0.08 and
  0.09, and the failure is sudden rather than gradual.

Between them is a band that works, and its width depends on the curvature of the
surface, which depends on the data — including its scale. There is no universal right answer, which is why the
learning rate is the first hyperparameter anyone tunes.

In [ ]:
# Optional plot. Skip if matplotlib is unavailable.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for lr in (0.0001, 0.001, 0.01, 0.05):
    hist, _ = run(lr, steps=100)
    ax[0].plot(hist, label=f"lr = {lr}")
ax[0].set(xlabel="step", ylabel="loss", yscale="log", title="learning rate")
ax[0].legend(); ax[0].grid(alpha=.3)

ws = np.linspace(-1, 5, 200)
bs = np.linspace(-4, 4, 200)
Z = np.array([[L(w, b) for w in ws] for b in bs])
cs = ax[1].contour(ws, bs, Z, levels=np.logspace(-1, 2.5, 14))
path = [(0.0, 0.0)]
w, b = 0.0, 0.0
for _ in range(60):
    g = grad(w, b); w, b = w - 0.02 * g[0], b - 0.02 * g[1]; path.append((w, b))
path = np.array(path)
ax[1].plot(path[:, 0], path[:, 1], "o-", ms=3, lw=1)
ax[1].set(xlabel="w", ylabel="b", title="the walk downhill (lr = 0.02)")
plt.tight_layout(); plt.show()

## Convergence — how many epochs?

"Converged" means the update term has gone to zero. The update term is `lr * gradient`, and
`lr` is constant, so it means the **gradient** has gone to zero — you are standing where
the surface is flat.

In [ ]:
w, b, lr = 0.0, 0.0, 0.02
print(f"{'step':>6}{'loss':>12}{'|gradient|':>14}{'update size':>14}")
for i in range(1, 601):
    g = grad(w, b)
    w, b = w - lr * g[0], b - lr * g[1]
    if i in (1, 10, 50, 100, 300, 600):
        print(f"{i:>6}{L(w, b):>12.6f}{np.linalg.norm(g):>14.6f}{lr * np.linalg.norm(g):>14.8f}")
print(f"\nfinal w = {w:.4f}, b = {b:.4f}")

The gradient shrinks as the minimum approaches, so **the steps shrink automatically** even
with a fixed learning rate. That is why a constant `lr` does not overshoot at the end: the
thing being multiplied by it is already small.

In practice nobody watches this. You run a fixed number of epochs — 100 to 1000 is a
typical starting range — and check the loss curve afterwards.

## The whole answer, in one paragraph

The loss is a surface over the parameters. At your current point the gradient says which
way is uphill, for each parameter separately. Subtracting it moves downhill. The learning
rate says how far to move, too small wastes time and too large diverges, and the gradient
naturally shrinks near the bottom so the steps do too. **Backpropagation is not the reason
this works — it is the fast way to get the gradient that makes it work.**

## Try it yourself

1. Find the largest learning rate that still converges here, to three decimals. Then double
   the size of `X` (multiply by 10) and find it again. What happened, and why is feature
   scaling in the same conversation?
2. Start from `(w, b) = (10, 10)` and re-run the sweep. Does the safe range of learning
   rates change?
3. Replace MSE with MAE inside `L` and re-run Concept 4. Is the sign argument still valid
   at `w = 2` exactly? (Look at what happens to the derivative at zero error.)
4. Add a second feature so the loss depends on three parameters, and confirm the "step
   against the gradient" cell still holds. Nothing about the argument used the fact that
   there were only two.